### Init Spark NLP

In [1]:
import json
import os

from google.colab import files

license_keys = files.upload()

with open(list(license_keys.keys())[0]) as f:
    license_keys = json.load(f)

# Defining license key-value pairs as local variables
locals().update(license_keys)

# Adding license key-value pairs to environment variables
os.environ.update(license_keys)

Saving spark_nlp_for_healthcare_spark_ocr_10849.json to spark_nlp_for_healthcare_spark_ocr_10849.json


In [2]:
# Installing pyspark and spark-nlp
! pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Installing Spark NLP Healthcare
! pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION  --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Installing Spark NLP Display Library for visualization
! pip install -q spark-nlp-display

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 746.2/746.2 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.9/569.9 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.9/66.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 130.2 MB/s eta 0:00:00


In [3]:
import json
import os

import sparknlp
import sparknlp_jsl

from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline,PipelineModel
from pyspark.sql.types import StringType

import pandas as pd
pd.set_option('display.max_colwidth', 200)

import warnings
warnings.filterwarnings('ignore')

params = {"spark.driver.memory":"16G",
          "spark.kryoserializer.buffer.max":"2000M",
          "spark.driver.maxResultSize":"2000M"}

print("Spark NLP Version :", sparknlp.version())
print("Spark NLP_JSL Version :", sparknlp_jsl.version())

spark = sparknlp_jsl.start(license_keys['SECRET'],params=params)

spark

Spark NLP Version : 6.3.2
Spark NLP_JSL Version : 6.3.0


### Process Data

In [4]:
data_path = "/content/drive/MyDrive/Losanna-METATRON/Code/recommendation_steps_v8_iteration_0.csv"
data = pd.read_csv(data_path, encoding="utf-8", engine="python", sep=';')
data.head()

,Unnamed: 0,Model_Alone_Step 1,Model_Alone_Step 2,Model_Alone_Step 3,Model_Studies_Step 1,Model_Studies_Step 2,Model_Studies_Step 3,Model_Workflow_Step 1,Model_Workflow_Step 2,Model_Workflow_Step 3,Model_Workflow_Studies_Step 1,Model_Workflow_Studies_Step 2,Model_Workflow_Studies_Step 3,Expert_Step 1,Expert_Step 2,Expert_Step 3
0,1,"Step 1 – Oncological strategy \n- Immediate assessment of the obstructive left‑colon lesion: if clinically stable, place an endoscopic self‑expanding metal stent to relieve obstruction; if not fe...",Step 2 – Essential missing data for each category \nA) Patient Assessment: none (no essential data lacking that would preclude formulating a reasonable plan). \nB) Primary Tumor Status: none (th...,"Step 3 – Anatomical resectability \nCompliant, provided that: \n- Future liver remnant volume is ≥30 % of total liver volume (or ≥40 % if underlying liver disease). \n- Pre‑operative liver func...",Step 1 – Oncological strategy \n- Confirm baseline staging with high‑quality liver MRI (or contrast‑enhanced CT) and chest imaging (CT or PET‑CT) to rule out occult extra‑hepatic disease. \n- Pe...,"Step 2 – Essential missing data for each category \nA) Patient Assessment \n- Performance status (ECOG or ASA). \n- Cardiopulmonary reserve (e.g., recent ECG, pulmonary function tests if indica...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Future liver remnant volume is ≥ 20‑30 % of total liver volume and liver function is adequate. \n- No invasion of major hepatic...",Step 1 – Oncological strategy \n- Urgent surgical resection of the obstructing left‑colon primary tumor (segmental colectomy with oncologic lymphadenectomy) to relieve obstruction and obtain path...,"Step 2 – Essential missing data for each category \nA) Patient Assessment: \n • ECOG/Performance status or ASA classification \n • Cardiopulmonary evaluation (ejection fraction, pulmonary f...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Vascular involvement is compatible with safe resection (no invasion of major portal or hepatic veins) \n- Biliary drainage of t...","Step 1 – Oncological strategy \n- Convene a multidisciplinary tumor board (colorectal surgeon, hepatobiliary surgeon, medical oncologist, radiologist, anesthesiologist). \n- Because the primary ...","Step 2 – Essential missing data for each category \nA. Patient Assessment \n- Current performance status / ECOG or ASA classification. \n- Cardiopulmonary reserve (e.g., recent ECG, echocardiog...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Vascular imaging confirms no invasion of the portal or hepatic veins supplying the future liver remnant. \n- Biliary anatomy sh...",Step 1 – Oncological strategy\nA. Primary tumor management\n•\tSymptomatic subocclusive left colon cancer.\n•\tRecommended approach: upfront oncologic left colectomy with lymphadenectomy.\n•\tTemp...,Step 2 – Missing data\nA. Patient assessment\n•\tPatient’s current performance status and comorbidities.\n•\tPatient’s surgical fitness for major procedures.\n•\tTreatment tolerance\nB. Primary Tu...,"Step 3 – Resectability assessment\nAnatomical Compliance\n•\tTwo small metastases, 2.2 and 3 cm.\n•\tBilobar but peripheral distribution.\n•\tNo reported vascular or biliary invasion.\n•\tAdequate..."
1,2,Step 1 – Oncological strategy \n- Confirm histology and stage of the primary colon lesion with colonoscopy and biopsy; assess depth of invasion and nodal status (CT‑based or endoscopic ultrasound...,Step 2 – Essential missing data for each category \nA) Patient Assessment: none (no essential data missing that would preclude formulating a reasonable recommendation). \nB) Primary Tumor Status...,"Step 3 – Anatomically compliant, provided that: \n- Future liver remnant volume is ≥30 % (or ≥40 % if underlying liver disease is present). \n- No vascular involvement of the right hepatic vein,...",Step 1 – Oncological strategy \n- Confirm diagnosis of the primary right‑colon

In [5]:
data.insert(1, "Case_Number", data.iloc[:,0])
data.head()

,Unnamed: 0,Case_Number,Model_Alone_Step 1,Model_Alone_Step 2,Model_Alone_Step 3,Model_Studies_Step 1,Model_Studies_Step 2,Model_Studies_Step 3,Model_Workflow_Step 1,Model_Workflow_Step 2,Model_Workflow_Step 3,Model_Workflow_Studies_Step 1,Model_Workflow_Studies_Step 2,Model_Workflow_Studies_Step 3,Expert_Step 1,Expert_Step 2,Expert_Step 3
0,1,1,"Step 1 – Oncological strategy \n- Immediate assessment of the obstructive left‑colon lesion: if clinically stable, place an endoscopic self‑expanding metal stent to relieve obstruction; if not fe...",Step 2 – Essential missing data for each category \nA) Patient Assessment: none (no essential data lacking that would preclude formulating a reasonable plan). \nB) Primary Tumor Status: none (th...,"Step 3 – Anatomical resectability \nCompliant, provided that: \n- Future liver remnant volume is ≥30 % of total liver volume (or ≥40 % if underlying liver disease). \n- Pre‑operative liver func...",Step 1 – Oncological strategy \n- Confirm baseline staging with high‑quality liver MRI (or contrast‑enhanced CT) and chest imaging (CT or PET‑CT) to rule out occult extra‑hepatic disease. \n- Pe...,"Step 2 – Essential missing data for each category \nA) Patient Assessment \n- Performance status (ECOG or ASA). \n- Cardiopulmonary reserve (e.g., recent ECG, pulmonary function tests if indica...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Future liver remnant volume is ≥ 20‑30 % of total liver volume and liver function is adequate. \n- No invasion of major hepatic...",Step 1 – Oncological strategy \n- Urgent surgical resection of the obstructing left‑colon primary tumor (segmental colectomy with oncologic lymphadenectomy) to relieve obstruction and obtain path...,"Step 2 – Essential missing data for each category \nA) Patient Assessment: \n • ECOG/Performance status or ASA classification \n • Cardiopulmonary evaluation (ejection fraction, pulmonary f...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Vascular involvement is compatible with safe resection (no invasion of major portal or hepatic veins) \n- Biliary drainage of t...","Step 1 – Oncological strategy \n- Convene a multidisciplinary tumor board (colorectal surgeon, hepatobiliary surgeon, medical oncologist, radiologist, anesthesiologist). \n- Because the primary ...","Step 2 – Essential missing data for each category \nA. Patient Assessment \n- Current performance status / ECOG or ASA classification. \n- Cardiopulmonary reserve (e.g., recent ECG, echocardiog...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Vascular imaging confirms no invasion of the portal or hepatic veins supplying the future liver remnant. \n- Biliary anatomy sh...",Step 1 – Oncological strategy\nA. Primary tumor management\n•\tSymptomatic subocclusive left colon cancer.\n•\tRecommended approach: upfront oncologic left colectomy with lymphadenectomy.\n•\tTemp...,Step 2 – Missing data\nA. Patient assessment\n•\tPatient’s current performance status and comorbidities.\n•\tPatient’s surgical fitness for major procedures.\n•\tTreatment tolerance\nB. Primary Tu...,"Step 3 – Resectability assessment\nAnatomical Compliance\n•\tTwo small metastases, 2.2 and 3 cm.\n•\tBilobar but peripheral distribution.\n•\tNo reported vascular or biliary invasion.\n•\tAdequate..."
1,2,2,Step 1 – Oncological strategy \n- Confirm histology and stage of the primary colon lesion with colonoscopy and biopsy; assess depth of invasion and nodal status (CT‑based or endoscopic ultrasound...,Step 2 – Essential missing data for each category \nA) Patient Assessment: none (no essential data missing that would preclude formulating a reasonable recommendation). \nB) Primary Tumor Status...,"Step 3 – Anatomically compliant, provided that: \n- Future liver remnant volume is ≥30 % (or ≥40 % if underlying liver disease is present). \n- No vascular involvement of the right hepatic vein,...",Step 1 – Oncological strategy \n- Confirm diagnosis of the pri

In [6]:
data_to_use = data.iloc[:,1:]
data_to_use.head()

,Case_Number,Model_Alone_Step 1,Model_Alone_Step 2,Model_Alone_Step 3,Model_Studies_Step 1,Model_Studies_Step 2,Model_Studies_Step 3,Model_Workflow_Step 1,Model_Workflow_Step 2,Model_Workflow_Step 3,Model_Workflow_Studies_Step 1,Model_Workflow_Studies_Step 2,Model_Workflow_Studies_Step 3,Expert_Step 1,Expert_Step 2,Expert_Step 3
0,1,"Step 1 – Oncological strategy \n- Immediate assessment of the obstructive left‑colon lesion: if clinically stable, place an endoscopic self‑expanding metal stent to relieve obstruction; if not fe...",Step 2 – Essential missing data for each category \nA) Patient Assessment: none (no essential data lacking that would preclude formulating a reasonable plan). \nB) Primary Tumor Status: none (th...,"Step 3 – Anatomical resectability \nCompliant, provided that: \n- Future liver remnant volume is ≥30 % of total liver volume (or ≥40 % if underlying liver disease). \n- Pre‑operative liver func...",Step 1 – Oncological strategy \n- Confirm baseline staging with high‑quality liver MRI (or contrast‑enhanced CT) and chest imaging (CT or PET‑CT) to rule out occult extra‑hepatic disease. \n- Pe...,"Step 2 – Essential missing data for each category \nA) Patient Assessment \n- Performance status (ECOG or ASA). \n- Cardiopulmonary reserve (e.g., recent ECG, pulmonary function tests if indica...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Future liver remnant volume is ≥ 20‑30 % of total liver volume and liver function is adequate. \n- No invasion of major hepatic...",Step 1 – Oncological strategy \n- Urgent surgical resection of the obstructing left‑colon primary tumor (segmental colectomy with oncologic lymphadenectomy) to relieve obstruction and obtain path...,"Step 2 – Essential missing data for each category \nA) Patient Assessment: \n • ECOG/Performance status or ASA classification \n • Cardiopulmonary evaluation (ejection fraction, pulmonary f...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Vascular involvement is compatible with safe resection (no invasion of major portal or hepatic veins) \n- Biliary drainage of t...","Step 1 – Oncological strategy \n- Convene a multidisciplinary tumor board (colorectal surgeon, hepatobiliary surgeon, medical oncologist, radiologist, anesthesiologist). \n- Because the primary ...","Step 2 – Essential missing data for each category \nA. Patient Assessment \n- Current performance status / ECOG or ASA classification. \n- Cardiopulmonary reserve (e.g., recent ECG, echocardiog...","Step 3 – Anatomical resectability \nCompliant, provided that: \n- Vascular imaging confirms no invasion of the portal or hepatic veins supplying the future liver remnant. \n- Biliary anatomy sh...",Step 1 – Oncological strategy\nA. Primary tumor management\n•\tSymptomatic subocclusive left colon cancer.\n•\tRecommended approach: upfront oncologic left colectomy with lymphadenectomy.\n•\tTemp...,Step 2 – Missing data\nA. Patient assessment\n•\tPatient’s current performance status and comorbidities.\n•\tPatient’s surgical fitness for major procedures.\n•\tTreatment tolerance\nB. Primary Tu...,"Step 3 – Resectability assessment\nAnatomical Compliance\n•\tTwo small metastases, 2.2 and 3 cm.\n•\tBilobar but peripheral distribution.\n•\tNo reported vascular or biliary invasion.\n•\tAdequate..."
1,2,Step 1 – Oncological strategy \n- Confirm histology and stage of the primary colon lesion with colonoscopy and biopsy; assess depth of invasion and nodal status (CT‑based or endoscopic ultrasound...,Step 2 – Essential missing data for each category \nA) Patient Assessment: none (no essential data missing that would preclude formulating a reasonable recommendation). \nB) Primary Tumor Status...,"Step 3 – Anatomically compliant, provided that: \n- Future liver remnant volume is ≥30 % (or ≥40 % if underlying liver disease is present). \n- No vascular involvement of the right hepatic vein,...",Step 1 – Oncological strategy \n- Confirm diagnosis of the primary right‑colo

#### Comparison Analysis

In [7]:
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

sentence_detector = SentenceDetector()\
    .setInputCols(["document"])\
    .setOutputCol("sentence")

tokenizer = Tokenizer()\
    .setInputCols(["sentence"])\
    .setOutputCol("token")

# labels = ["Adenopathy", "Age","Biomarker","Biomarker_Result","Body_Part","Cancer_Dx","Cancer_Surgery",
#     "Cycle_Count","Cycle_Day","Date","Death_Entit","Directio","Dosage","Duration","Frequency",
#     "Gender","Grade","Histological_Type","Imaging_Test","Invasion","Metastasis","Oncogene","Pathology_Test",
#     "Race_Ethnicity","Radiation_Dose","Relative_Date","Response_To_Treatment","Route","Smoking_Status",
#     "Staging","Therapy","Tumor_Finding","Tumor_Size"]
labels = ["Biomarker","Biomarker_Result", "Cancer_Surgery", "Grade","Histological_Type","Imaging_Test","Invasion", "Metastasis", "Oncogene", "Response_To_Treatment",
          "Staging","Therapy","Tumor_Finding","Tumor_Size"]
# labels = ["Cancer_Surgery", "Imaging_Test", "Therapy"]

pretrained_zero_shot_ner = PretrainedZeroShotNER().pretrained("zeroshot_ner_oncology_large", "en", "clinical/models")\
    .setInputCols("sentence", "token")\
    .setOutputCol("ner")\
    .setPredictionThreshold(0.7)\
    .setLabels(labels)

ner_converter = NerConverterInternal()\
    .setInputCols("sentence", "token", "ner")\
    .setOutputCol("ner_chunk")


pipeline = Pipeline().setStages([
    document_assembler,
    sentence_detector,
    tokenizer,
    pretrained_zero_shot_ner,
    ner_converter
])

zeroshot_ner_oncology_large download started this may take some time.
Approximate size to download 1.5 GB
[OK!]


In [8]:
data_to_use.columns

Index(['Case_Number', 'Model_Alone_Step 1', 'Model_Alone_Step 2',
       'Model_Alone_Step 3', 'Model_Studies_Step 1', 'Model_Studies_Step 2',
       'Model_Studies_Step 3', 'Model_Workflow_Step 1',
       'Model_Workflow_Step 2', 'Model_Workflow_Step 3',
       'Model_Workflow_Studies_Step 1', 'Model_Workflow_Studies_Step 2',
       'Model_Workflow_Studies_Step 3', 'Expert_Step 1', 'Expert_Step 2',
       'Expert_Step 3'],
      dtype='object')

In [ ]:
# For step 2
compare_data_dict = {"Model_Alone":data_to_use.loc[:,["Case_Number","Model_Alone_Step 2","Expert_Step 2"]],
                     "Model_Studies":data_to_use.loc[:,["Case_Number","Model_Studies_Step 2","Expert_Step 2"]],
                     "Model_Workflow":data_to_use.loc[:, ["Case_Number","Model_Workflow_Step 2","Expert_Step 2"]],
                     "Model_Workflow_Studies":data_to_use.loc[:, ["Case_Number","Model_Workflow_Studies_Step 2","Expert_Step 2"]]}

In [9]:
# For step 3
compare_data_dict = {"Model_Alone":data_to_use.loc[:,["Case_Number","Model_Alone_Step 3","Expert_Step 3"]],
                     "Model_Studies":data_to_use.loc[:,["Case_Number","Model_Studies_Step 3","Expert_Step 3"]],
                     "Model_Workflow":data_to_use.loc[:, ["Case_Number","Model_Workflow_Step 3","Expert_Step 3"]],
                     "Model_Workflow_Studies":data_to_use.loc[:, ["Case_Number","Model_Workflow_Studies_Step 3","Expert_Step 3"]]}

In [10]:
compare_data_dict["Model_Alone"].iloc[7,:]

,7
Case_Number,8
Model_Alone_Step 3,"Step 3 – Anatomically compliant, provided that: \n- Volumetric analysis confirms an adequate future liver remnant (≥30 % in a normal liver, ≥40 % if steatosis/fibrosis). \n- Imaging shows the se..."
Expert_Step 3,Step 3 – Resectability assessment\nAnatomical Compliance\n•\tSingle 2.7 cm segment IVb metastasis.\n•\tAnatomically favorable for limited resection.\n•\tNo vascular or biliary invasion reported.\n...


In [12]:
compare_data_dict.keys()

dict_keys(['Model_Alone', 'Model_Studies', 'Model_Workflow', 'Model_Workflow_Studies'])

In [13]:
for approach in compare_data_dict.keys():

    print(f"Approach in process: {approach}")
    compare_data = compare_data_dict[approach]
    all_case_entities = {c:{"Text":{}, "Ref_Text":{}} for c in compare_data["Case_Number"].values}
    complete_df = pd.DataFrame(columns=["Case_N", "Source", "Entity", "Entity_Label", "First_Position_Span"])

    text_col = f"{approach}_Step 3"
    ref_text_col = "Expert_Step 3"

    for i in range(0, len(compare_data)):
        case_n = compare_data.loc[i,"Case_Number"]
        print(f"Case in process: {case_n}")

        # Text
        text_df = spark.createDataFrame([[compare_data.loc[i,text_col]]]).toDF("text")
        text_result = pipeline.fit(text_df).transform(text_df)
        # Flat df for the analysis
        flat_text_result = text_result.select(
            F.explode(
                F.arrays_zip(
                    text_result.ner_chunk.result,
                    text_result.ner_chunk.begin,
                    text_result.ner_chunk.end,
                    text_result.ner_chunk.metadata
                )
            ).alias("cols")).select(F.expr("cols['0']").alias("chunk"),
                                F.expr("cols['1']").alias("begin"),
                                F.expr("cols['2']").alias("end"),
                                F.expr("cols['3']['entity']").alias("ner_label")).filter("ner_label!='O'")
        # Save entities
        current_text_entities = {l:[] for l in labels}
        for row in flat_text_result.toLocalIterator():
            entity = row.chunk
            label = row.ner_label
            if entity not in current_text_entities[label]:
              current_text_entities[label].append(entity)
              complete_df.loc[len(complete_df)] = [case_n, "Text", entity, label, [row.begin, row.end]]
        all_case_entities[case_n]["Text"] = current_text_entities

        # Ref Text
        ref_text_df = spark.createDataFrame([[compare_data.loc[i,ref_text_col]]]).toDF("text")
        ref_text_result = pipeline.fit(ref_text_df).transform(ref_text_df)
        ref_flat_text_result = ref_text_result.select(
            F.explode(
                F.arrays_zip(
                    ref_text_result.ner_chunk.result,
                    ref_text_result.ner_chunk.begin,
                    ref_text_result.ner_chunk.end,
                    ref_text_result.ner_chunk.metadata
                )
            ).alias("cols")).select(F.expr("cols['0']").alias("chunk"),
                                F.expr("cols['1']").alias("begin"),
                                F.expr("cols['2']").alias("end"),
                                F.expr("cols['3']['entity']").alias("ner_label")).filter("ner_label!='O'")

        # Save entities
        current_ref_text_entities = {l:[] for l in labels}
        for row in ref_flat_text_result.toLocalIterator():
            entity = row.chunk
            label = row.ner_label
            if entity not in current_ref_text_entities[label]:
                current_ref_text_entities[label].append(entity)
                complete_df.loc[len(complete_df)] = [case_n, "Ref_Text", entity, label, [row.begin, row.end]]
        all_case_entities[case_n]["Ref_Text"] = current_ref_text_entities

        # Convert keys in str
        entities = {}
        for k in all_case_entities.keys():
            entities[str(k)] = all_case_entities[k]

    print(f"Saving....DF len: {len(complete_df)}")
    with open(f"/content/drive/MyDrive/Losanna-METATRON/Code/entities_{approach}_Step3.json", "w") as f:
        json.dump(entities, f)
    complete_df.to_csv(f"/content/drive/MyDrive/Losanna-METATRON/Code/entities_df_{approach}_Step3.csv",index=False)

Approach in process: Model_Alone
Case in process: 1
Case in process: 2
Case in process: 3
Case in process: 4
Case in process: 5
Case in process: 6
Case in process: 7
Case in process: 8
Case in process: 9
Case in process: 10
Case in process: 11
Case in process: 12
Case in process: 13
Case in process: 14
Case in process: 15
Case in process: 16
Case in process: 17
Case in process: 18
Case in process: 19
Case in process: 20
Case in process: 21
Case in process: 22
Case in process: 23
Case in process: 24
Case in process: 25
Case in process: 26
Case in process: 27
Case in process: 28
Case in process: 29
Case in process: 30
Case in process: 31
Case in process: 32
Case in process: 33
Case in process: 34
Case in process: 35
Case in process: 36
Case in process: 37
Case in process: 38
Case in process: 39
Case in process: 40
Case in process: 41
Case in process: 42
Case in process: 43
Case in process: 44
Case in process: 45
Case in process: 46
Case in process: 47
Case in process: 48
Case in process: